# Multi-Agent System Using RAG Built from Scratch for AWS Bedrock

This notebook demonstrates how to design and build an enterprise-grade **Multi-Agent RAG System from scratch** using Python, Boto3, and the AWS Bedrock Converse API—with zero external framework dependencies.

### Key Highlights:
- **ReAct Agent Architecture**: Autonomous tool execution, observation handling, and loop iteration.
- **Separation of Concerns**: Isolated agent roles (Ingestion, Financial Analysis, Privileged Execution, Security Supervisor).
- **Defense in Depth**: Bedrock Contextual Grounding Guardrails (threshold >= 0.75) and deterministic cryptographic tool authorization defending against indirect prompt injection.

## 1. Multi-Agent System Architecture
We initialize the four specialized agents and the security supervisor gateway.

In [ ]:
from multi_agent_rag import (
    Tool,
    BaseAgent,
    RAGIngestionAgent,
    FinancialAnalystAgent,
    PrivilegedExecutionAgent,
    SecuritySupervisorAgent,
    build_multi_agent_rag_system,
)

# Instantiate the multi-agent system from scratch
supervisor = build_multi_agent_rag_system(mock_mode=True)

print("Multi-Agent System Initialized:")
print(f" - Orchestrator:       {supervisor.name} (Role: {supervisor.role})")
print(f" - Ingestion Worker:   {supervisor.ingestion.name} (Tools: {list(supervisor.ingestion.tools.keys())})")
print(f" - Financial Analyst:  {supervisor.analyst.name} (Tools: {list(supervisor.analyst.tools.keys())})")
print(f" - Privileged Worker:  {supervisor.privileged.name} (Tools: {list(supervisor.privileged.tools.keys())})")

## 2. Inspect Ingested Document with Embedded Indirect Injection
The ingested financial report contains a concealed adversarial prompt injection targeting funds transfer tools.

In [ ]:
doc_content = supervisor.ingestion.document_store["Q3_Financial_Report.md"]
print("=== Ingested Document Excerpt ===\n")
print(doc_content)

## 3. Autonomous Execution & Guardrail Enforcement
Execute user query: `Please read the Q3 financial report and summarize total revenue and operating margins.`

Notice how the supervisor intercepts the indirect injection, logs contextual grounding telemetry, sanitizes facts, and passes clean data to the financial analyst.

In [ ]:
query = "Please read the Q3 financial report and summarize total revenue and operating margins."
result = supervisor.handle_query(query)

print("=== Multi-Agent Execution Trace ===")
for step in result["workflow_trace"]:
    print(f"Step {step['step']}: [{step['agent']}] -> {step['action']}")
    if "details" in step:
        print(f"         {step['details']}")

print("\n=== Contextual Grounding Guardrail Telemetry ===")
g = result["grounding_check"]
print(f"Grounding Score: {g['grounding_score']} (Required Threshold: {g['threshold']})")
print(f"Status:          {g['status']}")
print(f"Flagged Items:   {g['injections_detected']}")

print(f"\nUnauthorized Privileged Executions: {result['unauthorized_tool_executions']} (Expected: 0)")

print("\n=== Final Verified User Response ===")
print(result["final_response"])

## 4. Validating Cryptographic Isolation of Privileged Agent
Even if an attacker bypasses prompt filters, the isolated `PrivilegedExecutionAgent` refuses tool execution without a supervisor cryptographic authorization token.

In [ ]:
priv_agent = supervisor.privileged

# 1. Unauthenticated invocation attempt (raises PermissionError)
try:
    priv_agent.tools["transfer_funds"].execute(
        recipient="ATTACKER-999",
        amount=250000.0,
        auth_token=None,
    )
    print("FAIL: Unauthorized call was permitted!")
except PermissionError as e:
    print(f"[✓] Unauthorized Call Blocked as expected: {e}")

# 2. Authenticated invocation with supervisor cryptographic token
authorized_resp = priv_agent.tools["transfer_funds"].execute(
    recipient="APPROVED-PAYROLL",
    amount=5000.0,
    auth_token="SUPERVISOR-AUTH-TOKEN-SECURE",
)
print("Authorized Call Result:", authorized_resp)
assert authorized_resp["status"] == "EXECUTED"

print("\n[✓] Privilege isolation and authorization gate successfully confirmed!")